# X4: Research Frontiers

Lessons 0 through 15 built, from first principles, the machinery underneath
essentially every model in production today. This closing notebook is a
map rather than another derivation: where does research go from here, what
is settled engineering versus still-open science, and where to keep
reading once this curriculum ends.

## Introduction

Four directions get a concrete, runnable demonstration below — not because
a toy notebook can reproduce frontier research at scale, but because each
one rests on a specific, checkable claim that a small experiment can make
visible: capacity predictably reduces error until something else becomes
the bottleneck, sparse computation can decouple parameter count from
per-token cost, aligning two representation spaces is a learnable
objective, and training measurably reorganises what a hidden layer
encodes. A closing section then points to where to track all of this as it
keeps moving.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, data generation) is reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
device = torch.device("cpu")

## Scaling Laws

**Scaling laws** are the empirical finding (Kaplan et al. 2020, Hoffmann
et al. 2022 / "Chinchilla") that a model's loss falls off as a predictable
power law in model size, data size, and compute — predictable enough to
extrapolate how much a much larger training run will improve *before*
running it. A small version of the same measurement is directly
reproducible: fix a 'student' architecture family and a fixed-size
training set, sweep capacity (hidden width), and check whether test loss
falls in a straight line on log-log axes.

In [ ]:
D_IN = 10
teacher = nn.Sequential(nn.Linear(D_IN, 40), nn.ReLU(), nn.Linear(40, 1))
for p in teacher.parameters():
    p.requires_grad = False  # a fixed, unknown-to-the-student ground truth function


def make_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, D_IN, generator=g)
    with torch.no_grad():
        y = teacher(X) + 0.05 * torch.randn(n, 1, generator=g)
    return X, y


X_train, y_train = make_data(2000, SEED)
X_test, y_test = make_data(500, SEED + 1)

widths = [2, 4, 8, 16, 32, 64]
test_losses = []
for w in widths:
    torch.manual_seed(1)
    student = nn.Sequential(nn.Linear(D_IN, w), nn.ReLU(), nn.Linear(w, 1))
    opt = optim.Adam(student.parameters(), lr=0.01)
    g = torch.Generator().manual_seed(0)
    for _ in range(500):
        batch = torch.randperm(2000, generator=g)[:256]
        loss = nn.functional.mse_loss(student(X_train[batch]), y_train[batch])
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        test_losses.append(nn.functional.mse_loss(student(X_test), y_test).item())

# Fit the power law only over the pre-plateau range -- the point past it is the demo's second finding.
log_w, log_l = np.log(widths[:4]), np.log(test_losses[:4])
slope, intercept = np.polyfit(log_w, log_l, 1)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(widths, test_losses, "o-", label="measured test loss")
ax.loglog(widths[:4], np.exp(intercept) * np.array(widths[:4]) ** slope, "k--",
          label=f"fitted power law, exponent={slope:.2f}")
ax.set_xlabel("student hidden width"); ax.set_ylabel("test MSE"); ax.legend()
ax.set_title("Capacity scaling on a fixed-size dataset")
plt.show()
print(f"widths {widths[:4]}: power-law exponent = {slope:.2f}")
print(f"widths {widths[4:]}: loss plateaus -- capacity stopped being the bottleneck")

Loss falls in a straight line on log-log axes exactly while capacity is
the limiting factor -- then plateaus once the student's width matches what
the fixed teacher function actually needs. This is the second, equally
important half of the scaling-laws finding: the power law holds only until
*something else* becomes the bottleneck (here, the task's true complexity;
at production scale, usually the size of the training data instead) --
which is exactly Chinchilla's headline result, that many large models of
the time were **over-parameterised relative to their training data**, and
a compute budget split more toward data than parameters reached a lower
loss for the same cost.

## Beyond Dense Transformers

10a's Transformer block runs every parameter on every token, and 10a's
attention pattern costs $O(n^2)$ in sequence length -- every position
attends to every other position. Two active research directions each
remove one of those costs.

**Mixture-of-Experts (MoE)** keeps the per-token compute of a small dense
layer while giving the model far more *total* parameters: a router picks a
small subset of "expert" sub-networks per token, and only those experts
run. Total capacity grows with the number of experts; per-token compute
does not.

In [ ]:
class MoELayer(nn.Module):
    def __init__(self, d_in, d_hidden, n_experts):
        super().__init__()
        self.experts = nn.ModuleList([nn.Linear(d_in, d_hidden) for _ in range(n_experts)])
        self.router = nn.Linear(d_in, n_experts)

    def forward(self, x):
        expert_idx = self.router(x).argmax(dim=-1)   # top-1 routing: each token picks one expert
        out = torch.zeros(x.shape[0], self.experts[0].out_features)
        for e, layer in enumerate(self.experts):
            mask = expert_idx == e
            if mask.any():
                out[mask] = layer(x[mask])
        return out, expert_idx


torch.manual_seed(SEED)
N_EXPERTS = 8
moe = MoELayer(d_in=16, d_hidden=32, n_experts=N_EXPERTS)
tokens = torch.randn(200, 16)
_, chosen_experts = moe(tokens)

total_expert_params = sum(p.numel() for p in moe.experts.parameters())
active_params_per_token = sum(p.numel() for p in moe.experts[0].parameters())
dense_equivalent_params = active_params_per_token  # a single dense layer of the same per-token size

print(f"total parameters across all {N_EXPERTS} experts: {total_expert_params:,}")
print(f"parameters actually used for any one token:      {active_params_per_token:,} ({active_params_per_token/total_expert_params:.1%} of total)")
print(f"a dense layer with the same per-token cost has:   {dense_equivalent_params:,} parameters total")
print(f"-> {N_EXPERTS}x the total capacity, at identical per-token compute")

**State-space models** (S4, Mamba) attack the other cost: they replace
attention's all-pairs comparison with a linear recurrence, so each new
token updates a fixed-size hidden state in $O(1)$ work rather than
comparing against every previous token -- $O(n)$ total instead of
attention's $O(n^2)$. The trade-off is real: a recurrent state must
compress the *entire* sequence history into a fixed size, while attention
can look back at any exact past token directly. The FLOP-count gap this
trades away is the direct reason the approach exists.

In [ ]:
d = 64
seq_lengths = np.array([128, 256, 512, 1024, 2048, 4096])
attention_flops = 2 * seq_lengths**2 * d          # O(n^2): every pair of positions compared
state_space_flops = 2 * seq_lengths * d           # O(n): one fixed-cost update per position

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(seq_lengths, attention_flops, "o-", label="attention: O(n^2)")
ax.loglog(seq_lengths, state_space_flops, "o-", label="state-space recurrence: O(n)")
ax.set_xlabel("sequence length"); ax.set_ylabel("FLOPs (analytic count)"); ax.legend()
ax.set_title("Why sequence length is the pressure point")
plt.show()
print(f"at n={seq_lengths[-1]}: attention needs {attention_flops[-1]/state_space_flops[-1]:.0f}x more FLOPs than the linear recurrence")

## Multimodal Models

8a built token embeddings for text; a multimodal model (CLIP and its
successors) learns embeddings for two *different* modalities -- an image
encoder and a text encoder -- into a **shared space**, trained so a matched
image/caption pair land close together and a mismatched pair land far
apart. Nothing about the underlying objective is modality-specific: a toy
version, with random 2-D vectors standing in for real image and text
encoders, makes the mechanism visible directly.

In [ ]:
def make_paired_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    concepts = torch.randn(n, 2, generator=g)         # the shared underlying "meaning"
    image_emb = concepts + 0.3 * torch.randn(n, 2, generator=g)
    text_emb = concepts + 0.3 * torch.randn(n, 2, generator=g)
    return image_emb, text_emb


image_emb, text_emb = make_paired_data(200, SEED)
image_proj = nn.Linear(2, 2)
text_proj = nn.Linear(2, 2)
opt = optim.Adam(list(image_proj.parameters()) + list(text_proj.parameters()), lr=0.05)

for step in range(300):
    z_img = image_proj(image_emb)
    z_txt = text_proj(text_emb)
    sim = z_img @ z_txt.T  # (n, n) matrix: sim[i, j] = how aligned image i is with text j
    targets = torch.arange(len(image_emb))              # the correct match for row i is column i
    loss = nn.functional.cross_entropy(sim, targets) + nn.functional.cross_entropy(sim.T, targets)
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    z_img, z_txt = image_proj(image_emb), text_proj(text_emb)
    sim = z_img @ z_txt.T
    top1_acc = (sim.argmax(dim=1) == torch.arange(len(image_emb))).float().mean().item()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].scatter(image_emb[:, 0], image_emb[:, 1], label="image", alpha=0.5)
axes[0].scatter(text_emb[:, 0], text_emb[:, 1], label="text", alpha=0.5)
axes[0].set_title("Before alignment"); axes[0].legend()
axes[1].scatter(z_img[:, 0], z_img[:, 1], label="image", alpha=0.5)
axes[1].scatter(z_txt[:, 0], z_txt[:, 1], label="text", alpha=0.5)
axes[1].set_title("After contrastive alignment"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"top-1 retrieval accuracy (correct text for a given image, out of {len(image_emb)}): {top1_acc:.1%}")

This is the exact same shape as 14a and 14b's objectives -- a differentiable
loss over a batch of paired examples, trained end-to-end -- applied to two
encoders instead of one. Production multimodal models scale the encoders
(a vision Transformer, a language Transformer) and the batch size (CLIP
used tens of thousands of pairs per batch, since more negatives make the
alignment task harder and the representation sharper), but the mechanism
is the one demonstrated above.

## Interpretability

**Mechanistic interpretability** asks what a trained network's internal
representations actually encode, rather than only whether its output is
correct. The standard tool is a **linear probe**: freeze the network,
train a small linear classifier on its hidden activations to predict some
property, and treat the probe's accuracy as evidence for whether that
property is *linearly* encoded there. A task that is not linearly
separable in the raw input -- XOR -- makes the effect of training
unmistakable: if a hidden layer becomes linearly probe-able for XOR, that
reorganisation happened during training, not before it.

In [ ]:
def make_xor_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    X = torch.rand(n, 2, generator=g) * 4 - 2
    y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).long()  # not linearly separable in (x1, x2) directly
    return X, y


X_train, y_train = make_xor_data(400, SEED)
X_test, y_test = make_xor_data(100, SEED + 1)

torch.manual_seed(SEED)
trained_net = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
opt = optim.Adam(trained_net.parameters(), lr=0.03)
for _ in range(400):
    loss = nn.functional.cross_entropy(trained_net(X_train), y_train)
    opt.zero_grad(); loss.backward(); opt.step()

untrained_net = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))  # same architecture, no training

with torch.no_grad():
    h_trained_train = trained_net[1](trained_net[0](X_train)).numpy()
    h_trained_test = trained_net[1](trained_net[0](X_test)).numpy()
    h_untrained_train = untrained_net[1](untrained_net[0](X_train)).numpy()
    h_untrained_test = untrained_net[1](untrained_net[0](X_test)).numpy()

probe_raw = LogisticRegression(max_iter=1000).fit(X_train.numpy(), y_train.numpy())
probe_trained = LogisticRegression(max_iter=1000).fit(h_trained_train, y_train.numpy())
probe_untrained = LogisticRegression(max_iter=1000).fit(h_untrained_train, y_train.numpy())

print(f"linear probe on raw (x1, x2):             {probe_raw.score(X_test.numpy(), y_test.numpy()):.1%}")
print(f"linear probe on an UNTRAINED hidden layer: {probe_untrained.score(h_untrained_test, y_test.numpy()):.1%}")
print(f"linear probe on the TRAINED hidden layer:  {probe_trained.score(h_trained_test, y_test.numpy()):.1%}")

The raw input probe sits at chance -- XOR genuinely is not linearly
separable there. The untrained network's random ReLU kinks already carve
the input space into pieces that help a little, but training pushes the
hidden representation to a point where the exact same class of linear
probe solves the task almost perfectly. This -- checking what a specific
probe can and cannot decode from a specific layer -- is the actual
day-to-day method behind headline interpretability results (finding that a
model linearly encodes board state, sentiment, or truthfulness in some
layer), just run here on a two-line synthetic task instead of a real
trained model.

## Staying Current

This curriculum builds the foundations; the field itself moves faster than
any notebook can. A short, concrete list of where to keep reading:

- **Venues**: NeurIPS, ICML, and ICLR are the three flagship peer-reviewed
  conferences; most new results appear on **arXiv** (categories `cs.LG`,
  `cs.CL`, `cs.CV`) months before any conference proceeding.
- **Leaderboards and papers**: Papers With Code links benchmark results
  directly to their source paper and code.
- **Reference texts**: Goodfellow, Bengio & Courville's *Deep Learning*
  for foundations; Bishop & Bishop's *Deep Learning: Foundations and
  Concepts* (2023) for a more current treatment; Jurafsky & Martin's
  *Speech and Language Processing* for NLP specifically.
- **Primary-source explainers**: Jay Alammar's *The Illustrated
  Transformer* and Andrej Karpathy's build-from-scratch video series cover
  exactly the derive-it-yourself approach this curriculum has followed,
  at production scale.
- **News and summaries**: newsletters like *Import AI* and *The Batch*
  track what shipped each week without requiring a full literature review.

The single most durable skill this curriculum aimed to build is not any
one of these facts -- it's the ability to open a new paper's method section
and recognise the derivation underneath it, because the underlying pieces
(gradients, attention, the ELBO, a loss function's actual objective) are
now first-principles knowledge rather than a black box.

## Key Takeaways

- **Scaling laws** hold as a power law only while the swept axis is the
  actual bottleneck -- measured here as a clean power-law region followed
  by a plateau once the student's capacity matched the task's true
  complexity, the same shape behind Chinchilla's compute-vs-data finding.
- **Mixture-of-experts** decouples total parameter count from per-token
  compute (measured here: 8x the parameters, identical per-token cost);
  **state-space models** decouple sequence-length cost from attention's
  $O(n^2)$ pattern, trading exact long-range lookup for a fixed-size state.
- **Multimodal alignment** is 14a/14b's contrastive-objective shape,
  applied to two encoders instead of one -- demonstrated here pulling
  matched image/text pairs together in a shared embedding space.
- **Linear probing** turns "what does this layer encode" into a
  measurable question: training measurably reorganised a hidden layer to
  make a not-linearly-separable task (XOR) linearly decodable.
- The specific facts in this notebook will age; the derivations from
  lessons 0-15 that let a new paper's claims be checked, not just read,
  will not.